<a href="https://colab.research.google.com/github/edina-s/ADVANCED-CSS/blob/master/01_Uvod_v_podatkovno_znanost/notebooks/01_prvi_stik_s_podatki/B_prvi_stik_s_podatki.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prvi stik s podatki — B: vaja (telekom)
**Akademija UI za poslovne aplikacije — Modul 1, Python ekosistem in spoznavanje s surovimi podatki**

Predavatelj je v zvezku `A_prvi_stik_s_podatki` postopek pokazal na spletni trgovini.
Zdaj ga ponovite sami na podatkih telekoma.

Celico zaženete s **Shift + Enter**.

### Ponovitev
- **DataFrame** = tabela: vrstica je opazovanje, stolpec je spremenljivka. En stolpec je **Series**.
- **Vektorizacija**: operacijo zapišemo za cel stolpec naenkrat — brez `for` zanke.
- Štirje ukazi za vsak nov nabor podatkov: `pd.read_csv()`, `.shape`, `.head()`, `.info()` / `.dtypes`.
- **Tip v Pandasu ni isto kot tip po pomenu.** Pandas ne ve, da je poštna številka oznaka in da je `odhod` skupina.
  Pogoste napake: datum kot besedilo (`object` / `str`), poštna številka kot število, celo število kot `float`.
- Cilj: **ena vrstica = eno opazovanje** (stranka, naročilo, …), en stolpec = ena lastnost.

### Nastavitev okolja
Zaženite spodnjo celico (izberite jo in pritisnite **Shift + Enter**). Ničesar ne spreminjajte.
Celica deluje lokalno in v Google Colabu.

In [1]:
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/valira-ai/ui-za-poslovne-aplikacije.git"

if "google.colab" in sys.modules:           # Google Colab: prenesemo gradivo z GitHuba
    KOREN = Path("/content/ui-za-poslovne-aplikacije")
    if not KOREN.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(KOREN)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "plotnine>=0.15", "kagglehub"], check=True)
else:                                       # lokalno: poiščemo mapo, v kateri je utils/orodja.py
    KOREN = Path.cwd()
    while not (KOREN / "utils/orodja.py").exists() and KOREN != KOREN.parent:
        KOREN = KOREN.parent
sys.path.insert(0, str(KOREN))              # da Python najde utils/orodja.py
print("Gradivo je v:", KOREN)

Gradivo je v: /content/ui-za-poslovne-aplikacije


### Podatki
Funkcija `poskrbi_za_podatke()` iz `utils/orodja.py` ob prvem zagonu prenese podatke s Kaggla (~1 minuta)
in jih shrani v mapo `data/`. Kako to deluje, smo pokazali v zvezku `A_prvi_stik_s_podatki`.

In [2]:
from utils.orodja import poskrbi_za_podatke
DATA = poskrbi_za_podatke()

Prenašam abhinav89/telecom-customer s Kaggla ...
Using Colab cache for faster access to the 'telecom-customer' dataset.
Prenašam olistbr/brazilian-ecommerce s Kaggla ...


100%|██████████| 42.6M/42.6M [00:00<00:00, 227MB/s]

Extracting files...


Podatki so pripravljeni v: /content/ui-za-poslovne-aplikacije/data


---
# Del B — Vaja: telekom
Podatki ameriškega mobilnega operaterja: **100 000 naročnikov**, za vsakega podatki o porabi, napravi, gospodinjstvu
in ali je **odšel** (`odhod = 1`). Izbrali smo 22 stolpcev in jih poimenovali po slovensko.

## B1. Naložite podatke
Zaženite celice. Ničesar ni treba spreminjati.

In [3]:
import numpy as np
import pandas as pd

telekom = pd.read_csv(DATA / "telekom.csv")
slovar = pd.read_csv(DATA / "telekom_slovar.csv")
slovar          # kaj pomeni kateri stolpec

,stolpec,izvirno_ime,opis
0,id_stranke,Customer_ID,Identifikator stranke
1,odhod,churn,Ali je stranka odšla (1) ali ostala (0)
2,mesecev_narocnine,months,"Število mesecev, odkar je stranka naročnik"
3,povp_mesecni_prihodek,rev_Mean,Povprečni mesečni prihodek od stranke (USD)
4,povp_mesecna_narocnina,totmrc_Mean,Povprečni fiksni mesečni strošek naročnine (USD)
5,povp_minute,mou_Mean,Povprečno število porabljenih minut na mesec
6,povp_minute_6m,avg6mou,Povprečne minute v zadnjih 6 mesecih
7,sprememba_minut,change_mou,Sprememba minut glede na pretekle 3 mesece (%)
8,povp_minute_prekoracitev,ovrmou_Mean,Povprečne minute nad vključenim paketom
9,povp_gostovanje,roam_Mean,Povprečno število klicev v gostovanju


In [4]:
telekom.shape

(100000, 22)

In [5]:
telekom.head()

,id_stranke,odhod,mesecev_narocnine,povp_mesecni_prihodek,povp_mesecna_narocnina,povp_minute,povp_minute_6m,sprememba_minut,povp_minute_prekoracitev,povp_gostovanje,...,stevilo_narocnin,stevilo_telefonov,dni_z_napravo,cena_naprave,splet_na_napravi,dualband,regija,zakonski_stan,dohodek_razred,stevilo_avtomobilov
0,1000001,1,61,23.9975,22.500,219.25,322.0,-157.25,0.00,0.0,...,2,2.0,361.0,149.98999,WCMB,Y,NORTHWEST/ROCKY MOUNTAIN AREA,S,4.0,3.0
1,1000002,0,56,57.4925,37.425,482.75,477.0,532.25,22.75,0.0,...,1,7.0,240.0,NaN,WC,N,CHICAGO AREA,S,5.0,1.0
2,1000003,1,58,16.9900,16.990,10.25,11.0,-4.25,0.00,0.0,...,1,2.0,1504.0,29.98999,NaN,N,GREAT LAKES AREA,M,5.0,2.0
3,1000004,0,60,38.0000,38.000,7.50,50.0,-1.50,0.00,0.0,...,1,1.0,1812.0,29.98999,NaN,N,CHICAGO AREA,M,6.0,1.0
4,1000005,0,57,55.2300,71.980,570.50,586.0,38.50,0.00,0.0,...,1,6.0,434.0,149.98999,WCMB,Y,NEW ENGLAND AREA,S,6.0,1.0


In [6]:
telekom.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 22 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   id_stranke                100000 non-null  int64  
 1   odhod                     100000 non-null  int64  
 2   mesecev_narocnine         100000 non-null  int64  
 3   povp_mesecni_prihodek     99643 non-null   float64
 4   povp_mesecna_narocnina    99643 non-null   float64
 5   povp_minute               99643 non-null   float64
 6   povp_minute_6m            97161 non-null   float64
 7   sprememba_minut           99109 non-null   float64
 8   povp_minute_prekoracitev  99643 non-null   float64
 9   povp_gostovanje           99643 non-null   float64
 10  povp_klici_podpora        100000 non-null  float64
 11  povp_prekinjeni_klici     100000 non-null  float64
 12  stevilo_narocnin          100000 non-null  int64  
 13  stevilo_telefonov         99999 non-null   fl

**Vprašanja** (odgovorite na glas ali si zapišite):
1. Koliko strank in koliko stolpcev imamo?
2. Kateri stolpci so tipa `object` / `str` (besedilo)? Je to smiselno?
3. Kateri stolpci so `float`, čeprav bi pričakovali cela števila? Zakaj?
4. Je vsaka stranka v tabeli samo enkrat? Namig: `telekom["id_stranke"].nunique()`

<details>
<summary><b>Rešitev (klikni)</b></summary>

1. 100 000 strank, 22 stolpcev.
2. `splet_na_napravi`, `dualband`, `regija`, `zakonski_stan` — to so kategorije, besedilo je smiselno.
3. `stevilo_telefonov`, `dohodek_razred`, `stevilo_avtomobilov`: v Pandasu (verzija 2) celoštevilski stolpec ne more vsebovati
   `NaN`, zato ga Pandas spremeni v `float`. **Float pogosto pomeni: nekaj manjka.**
   Pozor tudi: `id_stranke` in `odhod` sta številki, a z njima ne računamo.
4. Da: 100 000 različnih `id_stranke` → **ena vrstica = ena stranka**.

</details>

Preverimo trditvi iz rešitve: koliko vrednosti manjka v teh treh stolpcih in koliko je različnih strank?

In [8]:
print(telekom[["stevilo_telefonov", "dohodek_razred", "stevilo_avtomobilov"]].isna().sum())
print("Različnih id_stranke:", telekom["id_stranke"].nunique())

stevilo_telefonov          1
dohodek_razred         25436
stevilo_avtomobilov    49366
dtype: int64
Različnih id_stranke: 100000


## B2. Raziščite posamezne stolpce
En stolpec tabele je **Series**: `telekom["zakonski_stan"]`. Nad njim kličemo metode:
- `.dtype` — tip, `.nunique()` — število različnih vrednosti, `.isna().sum()` — število manjkajočih,
- `.value_counts(dropna=False)` — kolikokrat se pojavi katera vrednost (z `dropna=False` prešteje tudi `NaN`).

V spodnji celici **zamenjajte ime stolpca** (med narekovaji) in jo ponovno zaženite.
Poskusite npr. `"zakonski_stan"`, `"splet_na_napravi"`, `"dohodek_razred"`, `"regija"`.

In [9]:
STOLPEC = "zakonski_stan"      # ← zamenjajte

s = telekom[STOLPEC]
print("Tip v Pandasu:      ", s.dtype)
print("Različnih vrednosti:", s.nunique())
print("Manjkajočih (NaN):  ", s.isna().sum())
s.value_counts(dropna=False).head(8)

Tip v Pandasu:       object
Različnih vrednosti: 5
Manjkajočih (NaN):   1732


,count
zakonski_stan,
U,37333
M,31052
S,17627
B,7116
A,5140
NaN,1732


Isto kodo bomo potrebovali večkrat, zato jo **zapakiramo v funkcijo**. `def` ustvari funkcijo,
v oklepaju so vhodni podatki (tabela in ime stolpca). Kopija funkcije je v `utils/orodja.py`,
zato jo v naslednjih zvezkih samo uvozimo: `from utils.orodja import opis_stolpca`.

In [10]:
def opis_stolpca(df, stolpec):
    s = df[stolpec]
    print("Tip v Pandasu:      ", s.dtype)
    print("Različnih vrednosti:", s.nunique())
    print("Manjkajočih (NaN):  ", s.isna().sum())
    print(s.value_counts(dropna=False).head(8).to_string())

opis_stolpca(telekom, "dualband")

Tip v Pandasu:       object
Različnih vrednosti: 4
Manjkajočih (NaN):   1
dualband
Y      72265
N      23196
T       4316
U        222
NaN        1


**Vprašanje:** pri `zakonski_stan` je največ vrednosti `U`. Kaj mislite, kaj pomeni? Je to res kategorija?

<details>
<summary><b>Rešitev (klikni)</b></summary>

`U` pomeni *unknown* — **neznano**. To je manjkajoča vrednost, ki se skriva kot navadna kategorija,
zato je `isna()` ne najde. Podobno `UNKW` v `splet_na_napravi` in `U` v `dualband`.
Popoldne bomo takšne vrednosti iskali in obravnavali.

</details>

Preverimo: vse vrednosti teh treh stolpcev, skupaj z `NaN`:

In [11]:
for stolpec in ["zakonski_stan", "splet_na_napravi", "dualband"]:
    print(stolpec, telekom[stolpec].value_counts(dropna=False).to_dict())

zakonski_stan {'U': 37333, 'M': 31052, 'S': 17627, 'B': 7116, 'A': 5140, nan: 1732}
splet_na_napravi {'WCMB': 75733, 'WC': 13843, nan: 10189, 'UNKW': 235}
dualband {'Y': 72265, 'N': 23196, 'T': 4316, 'U': 222, nan: 1}


---
### Za hitre
Zaženite `telekom.describe().T` (`.T` obrne tabelo: en stolpec = ena vrstica) in poglejte:
1. Stolpec **min**: kje so negativne vrednosti? Je to mogoče?
2. Stolpca **75%** in **max**: kje je največja vrednost zelo daleč od ostalih?
3. Stolpec **count**: zakaj nimajo vsi stolpci 100 000 vrednosti?
4. Kaj pomeni povprečje (`mean`) pri `id_stranke`? In pri `odhod`?

Primerjajte lahko tudi `telekom["regija"].value_counts()` in `telekom["regija"].value_counts(dropna=False)`.